# 动手练习：网络分析流程与 NetworkX 实践

来自《生物医学网络科学》（Biomedical Network Science）。 参考答案版。

In [ ]:
# 数据文件（JupyterLite 专用；文件已在时什么也不做）
import os, js
from pyodide.http import pyfetch
_base = str(js.location.href).split("/extensions/")[0].split("/build/")[0] + "/files/practice/data/"
os.makedirs("data", exist_ok=True)
for _f in ['yeast_string_physical_v12_hc.csv', 'yeast_string_physical_v12_nodes.csv']:
    if not os.path.exists("data/" + _f):
        _r = await pyfetch(_base + _f)
        open("data/" + _f, "wb").write(await _r.bytes())


## 任务 1：读入边表、过滤、清洗，并报告网络概况

本练习使用真实数据：STRING v12.0 酿酒酵母（*Saccharomyces cerevisiae*）的物理相互作用子网络（CC BY 4.0，Szklarczyk 等 2023, *Nucleic Acids Research*）。文件 `data/yeast_string_physical_v12_hc.csv` 已只保留综合得分 combined score ≥ 700 的边，每对蛋白一行，列为 `protein1, protein2, combined_score`；`data/yeast_string_physical_v12_nodes.csv` 给出每个蛋白的基因名 `preferred_name`。来源与处理见 `data/SOURCES.md`。

为了让后面的计算在几秒内完成，这里使用更严格的阈值 **combined score ≥ 950**。请：

1. 用 pandas 读入边表，过滤得到 `edges_hc`（只含得分 ≥ 950 的行）；
2. 用 `nx.from_pandas_edgelist` 建无向图 `G`，把得分存为边属性 `combined_score`；
3. 按“先看数据，再算指标”的原则检查：自环数 `n_self`、连通分支数 `n_comp`；
4. 取最大连通分支 `G_lcc`（独立副本），并计算它占全部节点的比例 `frac_lcc`。

期望看到：约两千多个蛋白、一万多条边，但网络并不连通——除一个巨分支外还有上百个小分支（很多是只有两个蛋白的“孤对”）。后面的路径与社团分析只在巨分支上做，报告里要写明丢了多少节点。

> **提示：** `G.subgraph(max(nx.connected_components(G), key=len)).copy()`；不加 `.copy()` 得到的是只读视图。

In [ ]:
# 准备（直接运行）
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

edges = pd.read_csv("data/yeast_string_physical_v12_hc.csv")
info = pd.read_csv("data/yeast_string_physical_v12_nodes.csv")
gene = dict(zip(info["id"], info["preferred_name"]))   # STRING 编号 → 基因名
print("原始边表 %d 行，得分范围 %d–%d" % (len(edges), edges.combined_score.min(), edges.combined_score.max()))

In [ ]:
edges_hc = edges[edges.combined_score >= 950]
G = nx.from_pandas_edgelist(edges_hc, "protein1", "protein2", edge_attr="combined_score")
n_self = nx.number_of_selfloops(G)
comps = sorted(nx.connected_components(G), key=len, reverse=True)
n_comp = len(comps)
G_lcc = G.subgraph(comps[0]).copy()
frac_lcc = G_lcc.number_of_nodes() / G.number_of_nodes()
print("过滤后：%d 个蛋白，%d 条边；自环 %d 条" % (G.number_of_nodes(), G.number_of_edges(), n_self))
print("连通分支 %d 个；最大连通分支 %d 个蛋白（%.1f%%）、%d 条边；丢弃 %d 个蛋白"
      % (n_comp, G_lcc.number_of_nodes(), 100 * frac_lcc, G_lcc.number_of_edges(),
         G.number_of_nodes() - G_lcc.number_of_nodes()))
print("只有两个蛋白的分支：%d 个" % sum(len(c) == 2 for c in comps))

In [ ]:
# 核对（不用修改）
assert edges_hc is not None and G is not None and G_lcc is not None, "edges_hc、G、G_lcc 还有没赋值的"
assert edges_hc.combined_score.min() >= 950, "edges_hc 里还有得分低于 950 的行，检查过滤条件"
assert len(edges_hc) == 13275, "得分 ≥ 950 的边应为 13275 行；检查是 >= 还是 >"
assert not G.is_directed(), "G 应是无向图（nx.Graph），PPI 的边没有方向"
assert G.number_of_nodes() == 2264 and G.number_of_edges() == 13275, "G 应有 2264 个节点、13275 条边；检查建图时的列名"
assert all("combined_score" in d for _, _, d in G.edges(data=True)), "边上缺少 combined_score 属性：from_pandas_edgelist 要加 edge_attr"
assert n_self == 0, "本数据没有自环；用 nx.number_of_selfloops(G) 计数"
assert n_comp == 246, "连通分支应为 246 个；用 nx.number_connected_components(G) 或 len(list(nx.connected_components(G)))"
assert G_lcc.number_of_nodes() == 1186, "最大连通分支应有 1186 个节点；检查是否按 len 取最大"
assert not nx.is_frozen(G_lcc), "G_lcc 是只读视图，记得加 .copy()"
assert abs(frac_lcc - 1186 / 2264) < 1e-3, "frac_lcc 应约为 0.52 = 最大连通分支节点数 / G 的节点数"
print("✓ 任务 1 通过")

## 任务 2：基本指标与中心性——注意几组容易混淆的函数

在 `G_lcc` 上计算：

- 平均度 `k_mean`（$2M/N$）与密度 `dens`；
- 平均聚集系数 `C_avg`（`nx.average_clustering`，各节点局部聚集系数的平均）与传递性 `trans`（`nx.transitivity`，全网“闭合三元组比例”）；
- 度同配系数 `r_assort`；
- 度最高的 5 个蛋白的**基因名**列表 `top5_deg`，以及抽样介数（`k=200` 个源点、`seed=0`）最高的 5 个蛋白的基因名列表 `top5_bc`。

期望看到：平均聚集系数与传递性数值不同（两者定义不同，不能混用）；度最高的几乎都是核糖体蛋白，而介数最高的是另一批蛋白——它们连接着不同的复合物。思考：度同配系数为正，与“生物网络常为负”的印象不同，可能与 STRING 物理子网络按“同一复合物”记边有关（复合物内部成员彼此全连，度相近）。

> **提示：** `sorted(G_lcc.degree(), key=lambda x: -x[1])[:5]`；`nx.betweenness_centrality(G_lcc, k=200, seed=0)`；用 `gene[...]` 换成基因名。

In [ ]:
N, M = G_lcc.number_of_nodes(), G_lcc.number_of_edges()
k_mean = 2 * M / N
dens = nx.density(G_lcc)
C_avg = nx.average_clustering(G_lcc)
trans = nx.transitivity(G_lcc)
r_assort = nx.degree_assortativity_coefficient(G_lcc)
top5_deg = [gene[v] for v, _ in sorted(G_lcc.degree(), key=lambda x: -x[1])[:5]]
bc = nx.betweenness_centrality(G_lcc, k=200, seed=0)
top5_bc = [gene[v] for v, _ in sorted(bc.items(), key=lambda x: -x[1])[:5]]
print("平均度 %.2f，密度 %.4f，平均聚集系数 %.3f，传递性 %.3f，度同配 %.3f" % (k_mean, dens, C_avg, trans, r_assort))
print("度前 5：", top5_deg)
print("介数前 5（抽样）：", top5_bc)

deg = np.array([d for _, d in G_lcc.degree()])
vals, counts = np.unique(deg, return_counts=True)
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].bar(vals, counts / counts.sum())
ax[0].set_xlabel("k"); ax[0].set_ylabel("P(k)"); ax[0].set_title("linear")
ax[1].loglog(vals, counts / counts.sum(), "o", ms=3)
ax[1].set_xlabel("k"); ax[1].set_title("log-log")
plt.tight_layout()

In [ ]:
# 核对（不用修改）
assert None not in (k_mean, dens, C_avg, trans, r_assort, top5_deg, top5_bc), "还有变量是 None"
assert abs(k_mean - 17.08) < 0.05, "平均度应约为 17.1；公式是 2M/N，别忘了乘 2"
assert abs(dens - 0.01441) < 5e-4, "密度应约为 0.0144 = 2M/(N(N-1))；可直接用 nx.density"
assert abs(C_avg - 0.665) < 0.005, "平均聚集系数应约为 0.665；用 nx.average_clustering（不是 transitivity）"
assert abs(trans - 0.691) < 0.005, "传递性应约为 0.691；用 nx.transitivity（不是 average_clustering）"
assert abs(r_assort - 0.572) < 0.01, "度同配系数应约为 +0.57；用 nx.degree_assortativity_coefficient，且在 G_lcc 上算"
assert list(top5_deg) == ["RPS5", "RPS13", "ASC1", "RPS6B", "RPS7A"], "度前 5 应为 RPS5、RPS13、ASC1、RPS6B、RPS7A；检查是否按度从大到小排序、是否换成了基因名"
assert len(top5_bc) == 5 and len(set(top5_bc) & {"RPS31", "UBP8", "SNU13", "SAC3", "TAF14", "NUP159", "TRA1"}) >= 3, \
    "介数前 5 与参考结果重合太少；检查是否用了 betweenness_centrality、按值从大到小排序并换成基因名"
print("✓ 任务 2 通过")

## 任务 3：社团发现、模块度与结果的稳定性

在 `G_lcc` 上：

1. 用 `nx.community.louvain_communities(G_lcc, seed=1)` 得到划分 `comms`，计算社团数 `n_comm` 与模块度 `Q`；
2. 用种子 0–4 各跑一次 Louvain，把 5 个模块度存入列表 `Q_seeds`，把 5 个社团数存入 `n_seeds`；
3. 找出最大社团中度最高的 3 个蛋白的基因名 `hub_big`，看看这个社团大致对应什么复合物。

期望看到：$Q$ 约 0.77，社团数在不同种子下有小幅波动（26–27 个左右）而 $Q$ 几乎不变。报告时应写明种子与多次运行的波动范围。注意：单看 $Q$ 高并不能说明社团结构“显著”，下一个任务要与零模型比较。

> **提示：** `max(comms, key=len)` 取最大社团；`G_lcc.degree(nbunch)` 只返回指定节点的度。

In [ ]:
comms = nx.community.louvain_communities(G_lcc, seed=1)
n_comm = len(comms)
Q = nx.community.modularity(G_lcc, comms)
Q_seeds, n_seeds = [], []
for s in range(5):
    cs = nx.community.louvain_communities(G_lcc, seed=s)
    Q_seeds.append(nx.community.modularity(G_lcc, cs))
    n_seeds.append(len(cs))
big = max(comms, key=len)
hub_big = [gene[v] for v, _ in sorted(G_lcc.degree(big), key=lambda x: -x[1])[:3]]
print("seed=1：%d 个社团，Q = %.3f；最大社团 %d 个蛋白，枢纽 %s" % (n_comm, Q, len(big), hub_big))
print("种子 0–4：Q 范围 %.3f–%.3f，社团数 %s" % (min(Q_seeds), max(Q_seeds), n_seeds))

In [ ]:
# 核对（不用修改）
assert comms is not None and Q is not None and n_comm is not None, "comms、n_comm、Q 还没有赋值"
assert sum(len(c) for c in comms) == G_lcc.number_of_nodes(), "划分应覆盖 G_lcc 的全部节点；检查是否在 G_lcc 上运行"
assert n_comm == len(comms), "n_comm 应等于社团个数 len(comms)"
assert abs(Q - 0.766) < 0.02, "模块度应约为 0.77；用 nx.community.modularity(G_lcc, comms)，并固定 seed=1"
assert len(Q_seeds) == 5 and len(n_seeds) == 5, "Q_seeds 和 n_seeds 应各有 5 个值（种子 0–4）"
assert max(Q_seeds) - min(Q_seeds) < 0.03 and min(Q_seeds) > 0.7, "5 个种子的 Q 应都在 0.76–0.77 左右；检查每次是否重新运行 Louvain"
assert hub_big is not None and len(hub_big) == 3 and all(isinstance(x, str) for x in hub_big), "hub_big 应是 3 个基因名（字符串）"
print("✓ 任务 3 通过")

## 任务 4：与 ER 和保度零模型比较

指标高低要相对零模型而言。对 `G_lcc` 分别生成两种参照网络各 5 个（种子 0–4）：

- **ER**：`nx.gnm_random_graph(N, M, seed=s)`，只保留节点数和边数；
- **保度随机化**：复制 `G_lcc` 后做 `nx.double_edge_swap(H, nswap=5*M, max_tries=100*M, seed=s)`，保留每个节点的度。

在每个参照网络上计算平均聚集系数和 Louvain 模块度（`seed=s`），存为长度 5 的数组 `C_er`、`C_ds`、`Q_er`、`Q_ds`。再以保度零模型为参照计算 $z$ 分数 `z_C`、`z_Q`：$z=(X_{\text{real}}-\bar X_{\text{rand}})/s_{\text{rand}}$（`np.std` 默认 `ddof=0` 即可）。

期望看到：真实网络的 $C$ 与 $Q$ 都远高于两种零模型；保度零模型的 $C$ 比 ER 高（一部分聚集来自度序列本身），这正是它“更严格”的含义。零模型只有 5 个时 $z$ 分数很粗糙，正式分析应生成 20–100 个。

> **提示：** 真实值用任务 2、3 的 `C_avg` 与 `Q`。

In [ ]:
N, M = G_lcc.number_of_nodes(), G_lcc.number_of_edges()
C_er, C_ds, Q_er, Q_ds = [], [], [], []
for s in range(5):
    R = nx.gnm_random_graph(N, M, seed=s)
    C_er.append(nx.average_clustering(R))
    Q_er.append(nx.community.modularity(R, nx.community.louvain_communities(R, seed=s)))
    H = G_lcc.copy()
    nx.double_edge_swap(H, nswap=5 * M, max_tries=100 * M, seed=s)
    C_ds.append(nx.average_clustering(H))
    Q_ds.append(nx.community.modularity(H, nx.community.louvain_communities(H, seed=s)))
C_er, C_ds, Q_er, Q_ds = map(np.array, (C_er, C_ds, Q_er, Q_ds))
z_C = (C_avg - C_ds.mean()) / C_ds.std()
z_Q = (Q - Q_ds.mean()) / Q_ds.std()
print("C：真实 %.3f，ER %.3f ± %.3f，保度 %.3f ± %.3f；z = %.0f" % (C_avg, C_er.mean(), C_er.std(), C_ds.mean(), C_ds.std(), z_C))
print("Q：真实 %.3f，ER %.3f ± %.3f，保度 %.3f ± %.3f；z = %.0f" % (Q, Q_er.mean(), Q_er.std(), Q_ds.mean(), Q_ds.std(), z_Q))

In [ ]:
# 核对（不用修改）
assert len(C_er) == 5 and len(C_ds) == 5 and len(Q_er) == 5 and len(Q_ds) == 5, "四个数组都应有 5 个值（种子 0–4）"
assert np.all(C_er < 0.03), "ER 的平均聚集系数应接近密度（约 0.014）；检查 gnm_random_graph 的 N、M 是否取自 G_lcc"
assert np.all((C_ds > 0.04) & (C_ds < 0.15)), "保度零模型的 C 应约为 0.08：比 ER 高、远低于真实值；检查 double_edge_swap 是否作用在 G_lcc 的副本上"
assert np.all(Q_ds < Q - 0.2) and np.all(Q_er < Q - 0.2), "零模型的模块度应明显低于真实网络；检查是否在随机网络（而不是 G_lcc）上运行 Louvain"
assert C_ds.mean() > C_er.mean(), "保度零模型的 C 应高于 ER：一部分聚集来自度序列"
assert z_C is not None and z_Q is not None and z_C > 20 and z_Q > 20, "z 分数应为很大的正数；公式是 (真实值 − 零模型均值) / 零模型标准差"
print("✓ 任务 4 通过")

## 任务 5：开放小分析——结论对置信度阈值稳健吗？

网络构建的选择要做敏感性分析。请自选**至少 3 个**阈值（例如 700、800、900、950；文件中最低为 700），对每个阈值重复“过滤 → 建图 → 取最大连通分支 → 计算指标”，把结果整理成 DataFrame `sens`：每行一个阈值，至少包含列

- `N_lcc`：最大连通分支的节点数；
- `frac_lcc`：最大连通分支占全部节点的比例；
- `C`：平均聚集系数；
- `Q`：Louvain 模块度（固定种子）。

可以再加你关心的列（度同配、社团数、与保度零模型的 $z$ 分数……），并画一张“指标随阈值变化”的图。最后用两三句话回答：哪些结论在所有阈值下都成立？哪些数值随阈值明显变化？如果只报告一个阈值，可能误导读者的地方在哪里？

> **提示：** 把任务 1–3 的步骤写成一个函数 `summarize(th)`，返回一个字典，再用 `pd.DataFrame([...])` 汇总。阈值 700 时网络有四万多条边，全部指标约需几秒。

In [ ]:
def summarize(th, seed=1):
    H = nx.from_pandas_edgelist(edges[edges.combined_score >= th], "protein1", "protein2")
    lcc = H.subgraph(max(nx.connected_components(H), key=len)).copy()
    cs = nx.community.louvain_communities(lcc, seed=seed)
    return {"threshold": th, "N": H.number_of_nodes(), "M": H.number_of_edges(),
            "N_lcc": lcc.number_of_nodes(), "frac_lcc": lcc.number_of_nodes() / H.number_of_nodes(),
            "C": nx.average_clustering(lcc), "r": nx.degree_assortativity_coefficient(lcc),
            "Q": nx.community.modularity(lcc, cs), "n_comm": len(cs)}


sens = pd.DataFrame([summarize(th) for th in (700, 800, 900, 950)]).set_index("threshold")
print(sens.round(3).to_string())
fig, ax = plt.subplots(figsize=(4.5, 3))
for col in ("frac_lcc", "C", "Q", "r"):
    ax.plot(sens.index, sens[col], "o-", label=col)
ax.set_xlabel("combined score threshold"); ax.legend(fontsize=8)
plt.tight_layout()

In [ ]:
# 核对（不用修改）
assert isinstance(sens, pd.DataFrame), "sens 应是 pandas DataFrame"
assert len(sens) >= 3, "至少比较 3 个阈值"
for col in ("N_lcc", "frac_lcc", "C", "Q"):
    assert col in sens.columns, "sens 缺少列 %s" % col
assert sens["N_lcc"].between(2, 3384).all(), "N_lcc 应在 2 到 3384（文件中的蛋白总数）之间"
assert sens["frac_lcc"].between(0, 1).all(), "frac_lcc 是比例，应在 0–1 之间"
assert sens["C"].between(0, 1).all(), "平均聚集系数应在 0–1 之间"
assert sens["Q"].between(-0.5, 1).all(), "模块度应在 −0.5 到 1 之间"
print("✓ 任务 5 通过")